# PointMIL quickstart

Load a trained PointMIL model, classify one ATLAS-1 cell and plot the point-level interpretation.

Requirements: `pip install -e .[notebook]`, the ATLAS-1 dataset under `$POINTMIL_DATA/ATLAS-1` (see `scripts/download_data.sh atlas1`) and either internet access (the model is downloaded from the Hugging Face Hub, `pip install -e .[hub]`) or your own checkpoint (`outputs/.../best.pt`).

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch

from pointmil import load_checkpoint, prepare_cloud
from pointmil.data.atlas1 import ATLAS1, read_obj_vertices
from pointmil.hub import load_pretrained

DATA = Path(os.environ.get('POINTMIL_DATA', 'data'))
CKPT = None  # or the path of your own checkpoint, e.g. 'outputs/table2_atlas1_dgcnn_instance/seed42/best.pt'
if CKPT is None:
    model, info = load_pretrained('atlas1', 'dgcnn', 'instance')  # PointMIL(DGCNN), Table 2 ATLAS-1
    print(info['path'], info['metrics']['test'])
else:
    model = load_checkpoint(CKPT)['model'].eval()

## Load one cell

The public release contains surface meshes (`data/<serial>.obj`); the training clouds (`pointclouds_4096/<serial>.npy`) are used when present. `prepare_cloud` subsamples 1024 points and normalises to the unit ball exactly like the training loaders.

In [ ]:
import pandas as pd

meta = pd.read_csv(DATA / 'ATLAS-1' / 'metadata.csv')
serial = meta.iloc[0, 0]
npy = DATA / 'ATLAS-1' / 'pointclouds_4096' / f'{serial}.npy'
points = np.load(npy) if npy.exists() else read_obj_vertices(str(DATA / 'ATLAS-1' / 'data' / f'{serial}.obj'))
x = prepare_cloud(points[:, :3], num_points=1024, sampling='fps')
print(serial, meta.iloc[0, 1], points.shape, '->', tuple(x.shape))

## Predict and interpret

In [ ]:
with torch.no_grad():
    out = model(x)
classes = ATLAS1.classes
pred = int(out['bag_logits'].argmax(1))
probs = torch.softmax(out['bag_logits'], 1)[0]
importance = model.point_importance(x)[0].numpy()  # [1024], predicted class
print('prediction:', classes[pred], {c: round(float(p), 3) for c, p in zip(classes, probs)})

In [ ]:
import matplotlib.pyplot as plt

xyz = out['xyz'][0].numpy()
fig = plt.figure(figsize=(13, 4))
for i, c in enumerate(classes):
    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    imp_c = torch.softmax(out['interpretation'], 1)[0, i].numpy()
    sc = ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=imp_c, cmap='Reds', s=4)
    ax.set_title(c + (' (predicted)' if i == pred else ''), fontsize=10)
    ax.set_axis_off()
plt.colorbar(sc, ax=fig.axes, shrink=0.6)
plt.show()

## Perturbation curve

Remove the most important points step by step and watch the logit of the predicted class drop (AOPCR measures the area between this curve and the one for random removal).

In [ ]:
from pointmil.interpret import calculate_aopcr

aopcr, curve, random_curve = calculate_aopcr(model, x, n_random=3)
steps = np.linspace(0, 50, curve.shape[1])
plt.plot(steps, curve[0], label='most important first')
plt.plot(steps, random_curve[0], '--', label='random')
plt.xlabel('% points removed'); plt.ylabel('change of predicted-class logit'); plt.legend()
print('AOPCR', float(aopcr[0]))